# Q-MedAI Final Experiment

This notebook reproduces the FINAL EXPERIMENT methodology: a leakage-safe full-data comparison, a separate fair 150-row matched-data comparison, a 4-qubit/3-layer/100-iteration VQC, and a 150-row quantum kernel. It does not perform a sweep and does not claim GPU use, hardware execution, or quantum advantage.

In [ ]:
# First execution cell: locate the supplied CSV and print the exact path used.
from pathlib import Path

candidates = []
for root in (Path('/kaggle/input'), Path.cwd(), Path('/kaggle/working')):
    if root.exists():
        candidates.extend(root.glob('**/breast_cancer.csv'))
DATASET_PATH = next((path for path in candidates if path.is_file()), None)
if DATASET_PATH is None:
    raise FileNotFoundError(
        'breast_cancer.csv was not found. Add or upload the Q-MedAI dataset, then run this cell again.'
    )
print(f'DATASET PATH USED: {DATASET_PATH.resolve()}')


In [ ]:
# Locate uploaded Q-MedAI source, then install only any missing pinned runtime packages.
import importlib.metadata
import subprocess
import sys

source_files = []
for root in (Path('/kaggle/input'), Path.cwd(), Path('/kaggle/working')):
    if root.exists():
        source_files.extend(root.glob('**/src/final_experiment.py'))
if not source_files:
    raise FileNotFoundError(
        'Q-MedAI source files were not found. Add/upload the complete Q-MedAI project directory as a Kaggle dataset.'
    )
PROJECT_ROOT = source_files[0].parents[1]
sys.path.insert(0, str(PROJECT_ROOT))

required = ['numpy==2.5.2', 'pandas==3.0.5', 'scikit-learn==1.9.0', 'pennylane==0.45.1', 'matplotlib==3.11.1']
module_names = ['numpy', 'pandas', 'sklearn', 'pennylane', 'matplotlib']
to_install = []
for package, module in zip(required, module_names):
    expected = package.split('==')[1]
    try:
        installed = importlib.metadata.version(module if module != 'sklearn' else 'scikit-learn')
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != expected:
        to_install.append(package)
if to_install:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *to_install])

from src.final_experiment import run_final_experiment
print(f'PROJECT PATH USED: {PROJECT_ROOT}')
print('Backend note: PennyLane default.qubit is a classical simulator; enabling a Kaggle accelerator does not automatically accelerate it.')


## Methodology

The runner uses `random_state=42`, a stratified train/test split, B=0 and M=1 target encoding, and `SelectKBest(f_classif)` with four features. Full-data classical models and the VQC fit preprocessing on the full training split. The quantum kernel uses a stratified 150-row training subset. For the fair matched-data table, the exact same 150 raw rows receive a newly fitted preprocessing pipeline before the three classical models and quantum kernel train; the held-out test set never enters fitting.

In [ ]:
# This executes the final configuration once and saves JSON, CSV, config, plots, and reports.
# The output directory is separate from the uploaded project, so input artifacts are never overwritten.
OUTPUT_DIR = Path('/kaggle/working/q-medai-final-results') if Path('/kaggle/working').exists() else PROJECT_ROOT / 'kaggle_results'
final_payload = run_final_experiment(csv_path=DATASET_PATH, results_dir=OUTPUT_DIR)
print('FINAL EXPERIMENT STATUS: COMPLETED')
print(f'ARTIFACT DIRECTORY: {OUTPUT_DIR}')


In [ ]:
from IPython.display import Image, display
import pandas as pd

def result_table(records):
    columns = ['model', 'type', 'status', 'accuracy', 'precision', 'recall', 'specificity', 'f1', 'roc_auc', 'training_time_seconds']
    return pd.DataFrame(records).loc[:, columns]

def resolve_plot_path(path_value):
    path = Path(path_value)
    candidates = [path, PROJECT_ROOT / path, OUTPUT_DIR / 'plots' / path.name]
    return next((candidate.resolve() for candidate in candidates if candidate.is_file()), None)

print('FULL-DATA FINAL RESULTS')
display(result_table(final_payload['full_data_final_results']['models']))
print('MATCHED-DATA COMPARISON — 150 TRAINING SAMPLES FOR ALL MODELS')
display(result_table(final_payload['matched_data_comparison']['models']))
print('OBSERVED PERFORMANCE DIFFERENCE (not quantum advantage)')
print(final_payload['analysis']['matched_quantum_kernel_vs_best_classical'])
print(final_payload['analysis']['statistical_confidence'])
print('SAVED AND INLINE-DISPLAYED PLOTS')
for path in final_payload['plots']:
    resolved = resolve_plot_path(path)
    if resolved is None:
        print(f'PLOT NOT FOUND: {path}')
        continue
    print(resolved)
    display(Image(filename=str(resolved)))


## Judge interpretation and the most important remaining gap

The matched-data dashboard is the correct visual comparison because every displayed model uses the same training observations and the same held-out test set. The Quantum Kernel can be described as competitive and selectively stronger than Random Forest on sensitivity, F1, and ROC-AUC in the saved matched experiment; it does not beat the strongest classical model overall.

The most important missing evidence is **statistical robustness**, not another decorative graph. This run uses one dataset and one train/test split, with no repeated-seed confidence intervals or external clinical validation. The dataset is cross-sectional, so it supports malignant-class detection research but does not prove earlier-in-time diagnosis. PennyLane `default.qubit` is a classical simulator, not physical quantum hardware.